# 02주 · 연속·에너지·층류와 난류

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 동수역학 복습 / 층류·난류

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week02/CE272_W02.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week02) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 주제별 실습 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 원본 기름 관로의 층류 계산


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

<a id="s01"></a>

## 기름 관로 예제

**먼저 예측**: 동점성계수가 큰 기름의 흐름을 물의 난류 공식으로 계산해도 되는가?

**문제**: 비중 0.85인 기름이 직경 15 cm의 원형 관에 0.02 m³/s로 흐른다. 길이 100 m의 손실수두와 중심 유속, 중심에서 5 cm 떨어진 유속을 구하시오. 동점성계수는 6×10⁻⁴ m²/s이다.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
D, Q, L, nu, r = .15, .02, 100., 6e-4, .05
V = Q/hyd.area_pipe(D)
Re = hyd.reynolds(Q,D,nu)
f = hyd.darcy_factor(Re)
loss = hyd.pipe_loss(Q,L,D,nu)
u_max = 2*V
u_r = u_max*(1-(r/(D/2))**2)
# Hagen–Poiseuille 식을 이용한 독립 검산
loss_poiseuille = 128*nu*L*Q/(np.pi*hyd.G*D**4)
pd.Series({'V_mean_m_s':V,'Re':Re,'Darcy_f':f,'headloss_m':loss,
           'Poiseuille_m':loss_poiseuille,'u_center_m_s':u_max,'u_at_r_m_s':u_r})

**검토 문제**: 동점성계수만 물의 값으로 바꾼 경우에도 포물선 속도분포를 쓸 수 있는지 검토하시오. 계산을 바꾸기 전에 Reynolds 수와 가정을 확인하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


<a id="s02"></a>

## 난류 응력·혼합길이·와점성계수

**먼저 예측**: u′와 v′의 평균이 0이면 u′v′의 평균도 반드시 0인가?

**문제**: 물의 난류 흐름에서 밀도는 1,000 kg/m³, 평균유속 경사는 0.3 s⁻¹, 속도변동 곱의 평균은 −0.0070 m²/s²이다. Reynolds 응력, 혼합길이와 동점성 와점성계수를 구하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
rho, gradient, uv_mean = 1000., .3, -.0070
tau = -rho * uv_mean
mixing_length = np.sqrt(tau / (rho * gradient**2))
nu_t = -uv_mean / gradient
pd.Series({'Reynolds_stress_Pa': tau,
           'mixing_length_m': mixing_length,
           'kinematic_eddy_viscosity_m2_s': nu_t})


**검토 문제**: 속도변동 곱의 평균 부호와 운동량 전달 방향의 관계를 설명하시오. 평균유속 경사만 두 배가 되면 혼합길이와 동점성 와점성계수가 어떻게 바뀌는지 예측하고 확인하시오.

검산: 응력과 동점성 와점성계수의 단위 확인


## 학습 기록

평균속도·국소속도와 층류·난류 모형의 적용 범위를 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록


In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')